# Vendor Master Data Quality

Initial scope: SAP-style vendor master data.

Phase 1 focus:
- General vendor data
- Data profiling
- Deterministic Data Quality rules
- Duplicate and similarity analysis later

In [85]:
import pandas as pd
vendor_columns = [
    "LIFNR",
    "LAND1",
    "NAME1",
    "NAME2",
    "ORT01",
    "PSTLZ",
    "STRAS",
    "TELF1",
    "STCD1",
    "SPERR",
    "LOEVM"
]

len(vendor_columns)

11

In [140]:
vendor_data = {
    "LIFNR": [
        "100001", "100002", "100003", "100004", "100005",
        "100006", "100007", "100008", "100009", "100010",
        "100011", "100012"
    ],

"KTOKK": [
    "ZSUP", "ZSUP", "ZSUP", "ZSUP", "ZSUP",
    "ZSUP", "ZSUP", "ZSER", "ZSUP", "ZSER",
    "ZSUP", "ZSUP"
],

    "LAND1": [
        "AE", "AE", "AE", "OM", "AE",
        "AE", "AE", None, "AE", "OM",
        "AE", "AE"
    ],

    "NAME1": [
        "Gulf Trading LLC",
        "Gulf Trading L.L.C.",
        "Al Noor Services",
        "Muscat Industrial Co",
        None,
        "Desert Engineering LLC",
        "DESERT ENGINEERING L.L.C",
        "Emirates Technical Services",
        "Al Falah Trading",
        "Oman Equipment Services",
        "Future Tech Solutions",
        "Future Tech Solution LLC"
    ],

    "NAME2": [
        None,
        None,
        "Maintenance Division",
        None,
        None,
        None,
        None,
        None,
        "Abu Dhabi Branch",
        None,
        None,
        None
    ],

    "ORT01": [
        "Abu Dhabi",
        "Abu Dhabi",
        "Dubai",
        "Muscat",
        "Dubai",
        "Abu Dhabi",
        "ABU DHABI",
        "Sharjah",
        "AbuDhabi",
        "Muscat",
        "Dubai",
        "Dubai"
    ],

    "PSTLZ": [
        "12345",
        "12345",
        None,
        "112",
        "00000",
        "45678",
        "45678",
        "54321",
        "12A45",
        "113",
        "67890",
        "67890"
    ],

    "STRAS": [
        "Mussafah Industrial Area",
        "Mussafah Ind. Area",
        "Sheikh Zayed Road",
        "Ruwi Industrial Estate",
        None,
        "Street 10, Mussafah",
        "Street 10 Mussafah",
        "Industrial Area 4",
        "Hamdan Street",
        "Ghala Industrial Area",
        "Business Bay Tower 2",
        "Business Bay, Tower 2"
    ],

    "TELF1": [
        "+971501234567",
        "0501234567",
        "+971 55 222 3344",
        "+96899112233",
        None,
        "02-5556677",
        "+97125556677",
        "12345",
        "+971501111222",
        "+968 24 567890",
        "+971504445555",
        "0504445555"
    ],

    "STCD1": [
        "100234567890003",
        "100234567890003",
        "100345678900003",
        "OM1234567",
        None,
        "100456789010003",
        "100456789010003",
        "ABC123",
        "100567890120003",
        "OM9876543",
        "100678901230003",
        "100678901230003"
    ],

    "SPERR": [
        "", "", "", "", "",
        "", "X", "", "", "",
        "", ""
    ],

    "LOEVM": [
        "", "", "", "", "",
        "", "", "", "X", "",
        "", ""
    ]
}


raw_data = pd.DataFrame(vendor_data)
#print(vendor_raw.info())
data = raw_data.copy()

rule_catalog=pd.read_excel(r'/Users/vivek/Documents/Data Science Projects/data-quality-solution/DQ_Rule_Catalog.xlsx')
dq_results = pd.DataFrame( columns=["WORKSTREAM","OBJECT","OBJECT_KEY","OBJECT_KEY_VALUE","RULE_ID","DQ_DIMENSION","FIELD","ISSUE"])


In [143]:

def add_dq_results(data,dq_results, condition, workstream, object , key, field, rule_id, dq_dimension, issue):
    rule_results = data[condition][[key]].copy()
    #print(len(rule_results))
    if(len(rule_results)>0):
        rule_results.rename(columns={key: "OBJECT_KEY_VALUE"}, inplace=True)
        rule_results["WORKSTREAM"]=workstream
        rule_results["OBJECT"]=object
        rule_results["OBJECT_KEY"]=key
        rule_results["RULE_ID"]=rule_id
        rule_results["DQ_DIMENSION"] = dq_dimension
        rule_results["FIELD"] = field
        rule_results["ISSUE"]= issue
        dq_results = pd.concat([dq_results, rule_results], ignore_index = True)

    #print(rule_results)
    
    dq_results = dq_results.drop_duplicates(
            subset=[
                "WORKSTREAM",
                "OBJECT",
                "OBJECT_KEY_VALUE",
                "RULE_ID"
            ],
            keep="last"
        )

    return dq_results
    #print(dq_results)


In [145]:
standard_missing = rule_catalog[ (rule_catalog["RULE_CLASS"]=="STANDARD") & (rule_catalog["RULE_TYPE"]=="MISSING")  ]

#standard_missing
for row in standard_missing.itertuples():
    condition = data[row.FIELD].isna()
    dq_results = add_dq_results(data,dq_results,condition,row.WORKSTREAM,row.OBJECT,row.KEY,row.FIELD,row.RULE_ID,row.DQ_DIMENSION,row.ISSUE)
    
dq_results


,WORKSTREAM,OBJECT,OBJECT_KEY,OBJECT_KEY_VALUE,RULE_ID,DQ_DIMENSION,FIELD,ISSUE
1,PTP,Vendor Master,LIFNR,100005,DQ0001,Completeness,NAME1,Vendor name is missing
2,PTP,Vendor Master,LIFNR,100008,DQ0002,Completeness,LAND1,Country is missing
3,PTP,Vendor Master,LIFNR,100003,DQ0004,Completeness,PSTLZ,Postal code is missing
4,PTP,Vendor Master,LIFNR,100005,DQ0005,Completeness,STRAS,Street address is missing
5,PTP,Vendor Master,LIFNR,100005,DQ0006,Completeness,STCD1,Tax ID is missing
6,PTP,Vendor Master,LIFNR,100005,DQ0007,Completeness,TELF1,Telephone number is missing
